In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
print("Loading California Housing dataset...")
df = pd.read_csv("housing.csv")

FEATURE_NAMES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
                 "Population", "AveOccup", "Latitude", "Longitude"]

print(f"   Dataset shape: {df.shape}")
print(f"   Features: {FEATURE_NAMES}")
print(f"   Sample rows:\n{df.head(3)}\n")


Loading California Housing dataset...
   Dataset shape: (2000, 9)
   Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
   Sample rows:
     MedInc   HouseAge   AveRooms  AveBedrms   Population  AveOccup  \
0  6.037732  21.762431   7.593949   1.609685  1897.072002  3.929087   
1  4.124899   4.366502   6.306926   1.054259   287.295308  4.605260   
2  6.610195  18.789847  12.446700   2.093374   521.971835  4.373606   

    Latitude   Longitude     Price  
0  37.604899 -115.058924  2.783666  
1  36.957827 -114.354198  2.440305  
2  37.357583 -119.212983  3.515650  



In [ ]:
# Separate features (X) from the target variable (y)
X = df[FEATURE_NAMES]
y = df["Price"]

# Check for missing values (California Housing has none, but good practice)
missing = X.isnull().sum().sum()
print(f"Missing values: {missing}")

# Split into training (80%) and testing (20%) sets
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"   Train size: {X_train.shape[0]} rows")
print(f"   Test  size: {X_test.shape[0]} rows\n")

🔍 Missing values: 0
   Train size: 1600 rows
   Test  size: 400 rows



In [4]:
# Feature scaling — StandardScaler standardizes features to mean=0, std=1
# Important for Linear Regression; doesn't affect tree models but keeps things consistent
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # only transform test (no fitting!)


In [5]:
models = {
    "Linear Regression":    LinearRegression(),
    "Decision Tree":        DecisionTreeRegressor(random_state=42),
    "Random Forest":        RandomForestRegressor(n_estimators=100, random_state=42),
}

In [ ]:
print("Training and evaluating models...\n")
print(f"{'Model':<25} {'MAE':>8} {'MSE':>10} {'RMSE':>8}")
print("-" * 55)

results = {}

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test_scaled)

    # Calculate evaluation metrics
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    results[name] = {"model": model, "MAE": mae, "MSE": mse, "RMSE": rmse}
    print(f"{name:<25} {mae:>8.4f} {mse:>10.4f} {rmse:>8.4f}")

print()

🏋️  Training and evaluating models...

Model                          MAE        MSE     RMSE
-------------------------------------------------------
Linear Regression           0.3282     0.1828   0.4275
Decision Tree               0.3275     0.1756   0.4191
Random Forest               0.2263     0.0887   0.2978



In [ ]:
best_name = min(results, key=lambda k: results[k]["RMSE"])
best_model = results[best_name]["model"]
best_metrics = results[best_name]

print(f"Best model: {best_name}")
print(f"   MAE  = {best_metrics['MAE']:.4f}")
print(f"   MSE  = {best_metrics['MSE']:.4f}")
print(f"   RMSE = {best_metrics['RMSE']:.4f}\n")


✅ Best model: Random Forest
   MAE  = 0.2263
   MSE  = 0.0887
   RMSE = 0.2978



In [8]:
model_bundle = {
    "model":        best_model,
    "scaler":       scaler,
    "model_name":   best_name,
    "feature_names": FEATURE_NAMES,
    "metrics": {
        "MAE":  round(best_metrics["MAE"], 4),
        "MSE":  round(best_metrics["MSE"], 4),
        "RMSE": round(best_metrics["RMSE"], 4),
    }
}

In [9]:
with open("model.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Model saved to model.pkl")


Model saved to model.pkl
